# Amazon Product Data Cleaning and Preparation

## Project
Amazon Germany Wireless Headphones Price and Competitor Intelligence

## Purpose
This notebook loads the raw JSON response from the Amazon Scraper API, identifies the product records, cleans the relevant fields, creates business-focused calculated columns and saves a processed dataset for PostgreSQL and Power BI.

## Import the required libraries

In [1]:
import json
import re
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

print("Libraries imported successfully.")

Libraries imported successfully.


## Define the project folders

In [2]:
PROJECT_ROOT = Path.cwd().parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data folder:", RAW_DATA_DIR)
print("Processed data folder:", PROCESSED_DATA_DIR)

Project root: C:\Users\srika\Documents\Srikanth_Amazon_BI_Project
Raw data folder: C:\Users\srika\Documents\Srikanth_Amazon_BI_Project\data\raw
Processed data folder: C:\Users\srika\Documents\Srikanth_Amazon_BI_Project\data\processed


## Locate and Load the Raw JSON File

In [3]:
json_files = list(RAW_DATA_DIR.glob("*.json"))

if not json_files:
    raise FileNotFoundError(
        "No JSON files were found in the data/raw folder."
    )

latest_json_file = max(
    json_files,
    key=lambda file_path: file_path.stat().st_mtime
)

print("Number of JSON files found:", len(json_files))
print("Latest JSON file:", latest_json_file.name)

Number of JSON files found: 1
Latest JSON file: amazon_wireless_headphones_de_20260617_145203.json


## Load the JSON file

In [4]:
with open(latest_json_file, "r", encoding="utf-8") as file:
    raw_data = json.load(file)

print("JSON loaded successfully.")
print("Top-level Python type:", type(raw_data))

JSON loaded successfully.
Top-level Python type: <class 'dict'>


## Inspect the JSON Structure

In [5]:
if isinstance(raw_data, dict):
    print("Top-level keys:")
    
    for key in raw_data.keys():
        print("-", key)
else:
    print("The JSON response is not a dictionary.")

Top-level keys:
- page
- products
- html


## Show each top-level key and its data type

In [6]:
for key, value in raw_data.items():
    print(
        f"Key: {key} | "
        f"Type: {type(value).__name__} | "
        f"Length: {len(value) if hasattr(value, '__len__') else 'N/A'}"
    )

Key: page | Type: int | Length: N/A
Key: products | Type: list | Length: 22
Key: html | Type: str | Length: 0


## Automatically search for lists containing product records

In [7]:
def find_lists_in_json(data, path="root"):
    """
    Recursively search the JSON structure and report every list found.
    """
    found_lists = []

    if isinstance(data, dict):
        for key, value in data.items():
            new_path = f"{path}.{key}"
            found_lists.extend(find_lists_in_json(value, new_path))

    elif isinstance(data, list):
        found_lists.append(
            {
                "path": path,
                "length": len(data),
                "sample_type": type(data[0]).__name__ if data else "empty"
            }
        )

        for index, item in enumerate(data[:2]):
            found_lists.extend(
                find_lists_in_json(item, f"{path}[{index}]")
            )

    return found_lists


list_locations = find_lists_in_json(raw_data)

list_locations_df = pd.DataFrame(list_locations)

list_locations_df

,path,length,sample_type
0,root.products,22,dict


## Identify the Product List Automatically

In [8]:
COMMON_PRODUCT_KEYS = [
    "products",
    "results",
    "search_results",
    "organic_results",
    "organic",
    "items",
    "data"
]


def find_product_list(data):
    """
    Search recursively for a likely product list.
    """
    if isinstance(data, dict):
        for key, value in data.items():
            if (
                key.lower() in COMMON_PRODUCT_KEYS
                and isinstance(value, list)
                and len(value) > 0
                and isinstance(value[0], dict)
            ):
                return value, key

        for key, value in data.items():
            result = find_product_list(value)

            if result is not None:
                return result

    elif isinstance(data, list):
        if data and isinstance(data[0], dict):
            sample_keys = {
                str(key).lower()
                for key in data[0].keys()
            }

            product_signals = {
                "asin",
                "title",
                "name",
                "price",
                "rating",
                "url"
            }

            if sample_keys.intersection(product_signals):
                return data, "nested_list"

        for item in data:
            result = find_product_list(item)

            if result is not None:
                return result

    return None


product_result = find_product_list(raw_data)

if product_result is None:
    raise ValueError(
        "A product list could not be identified automatically. "
        "Review the JSON structure table above."
    )

product_list, product_list_key = product_result

print("Product list identified.")
print("Detected key:", product_list_key)
print("Number of product records:", len(product_list))

Product list identified.
Detected key: products
Number of product records: 22


## Inspect the first product record 

In [9]:
print(
    json.dumps(
        product_list[0],
        indent=2,
        ensure_ascii=False
    )
)

{
  "asin": "B0FSH42VZW",
  "title": "JBL Tune 730 BT Wireless Over-Ear-Kopfhörermit JBL Pure Bass Sound, Bis zu 76 Stunden Musikwiedergabe, Bluetooth 6.0, leichtem, faltbarem Design, Google Fast Pair, Microsoft Swift Pair,Schwarz",
  "url": "https://www.amazon.de/sspa/click?ie=UTF8&spc=MTo3MzE3NTIzMTI1MDM2MTE1OjE3ODE3MDA0NzQ6c3BfYXRmOjMwMDgwMzQ2NTQ1MDkzMjo6MDo6&url=%2FJBL-Over-Ear-Kopfh%25C3%25B6rermit-Musikwiedergabe-Bluetooth-faltbarem-Black%2Fdp%2FB0FSH42VZW%2Fref%3Dsr_1_1_sspa%3Fdib%3DeyJ2IjoiMSJ9.6bFymcQNAFwDinbk1Lz45Bv0tRYEFM2vgvb8mjd0V8iSD2biGxbjiJ5ckNzNWOXzhJrIIFGifwcLQfGY2N9LAZq_3pq-BthWysJKQs1sz8CJw0V_IkkkSEtG4Ev0o3gdwS9uYx3gVKr285TXnpwHdT_pj__UlZj0Y4KC-Porqh1RNzxgriSwIGuTMz0rOEie_thvKWYu-1HLR9-dJ3djTdN3NgTzH7rc9zhpT6cd6G4.8CM7y20NpJ9fSYQFaAG6oFLrf-eK50zqY5jGNxbRHt8%26dib_tag%3Dse%26keywords%3Dwireless%2Bheadphones%26qid%3D1781700474%26sr%3D8-1-spons%26aref%3DEqvqSWMZQZ%26sp_csd%3Dd2lkZ2V0TmFtZT1zcF9hdGY%26psc%3D1&aref=EqvqSWMZQZ&sp_cr=DUB",
  "price": 59.99,
  "price_strike

## Show all available product fields

In [10]:
all_product_fields = sorted(
    {
        key
        for product in product_list
        if isinstance(product, dict)
        for key in product.keys()
    }
)

print("Available product fields:")
for field in all_product_fields:
    print("-", field)

print("\nTotal available fields:", len(all_product_fields))

Available product fields:
- asin
- best_seller
- currency
- highest_price
- image
- is_amazons_choice
- is_prime
- is_sponsored
- is_video
- manufacturer
- organic_position
- price
- price_strikethrough
- pricing_count
- rating
- reviews_count
- sales_volume
- shipping_information
- sponsored_position
- title
- url

Total available fields: 21


# Convert the Products into a DataFrame

## Create the initial DataFrame

In [12]:
products_raw_df = pd.json_normalize(product_list)

print("DataFrame created successfully.")
print("Number of rows:", products_raw_df.shape[0])
print("Number of columns:", products_raw_df.shape[1])

products_raw_df.head()

DataFrame created successfully.
Number of rows: 22
Number of columns: 21


,asin,title,url,price,price_strikethrough,currency,rating,reviews_count,sales_volume,image,...,is_amazons_choice,is_sponsored,best_seller,organic_position,sponsored_position,shipping_information,highest_price,pricing_count,manufacturer,is_video
0,B0FSH42VZW,JBL Tune 730 BT Wireless Over-Ear-Kopfhörermit...,https://www.amazon.de/sspa/click?ie=UTF8&spc=M...,59.99,79.99,EUR,4.4,463,None,https://m.media-amazon.com/images/I/61WyJZWEhz...,...,False,False,False,1,None,None,79.99,None,None,False
1,B0FV3VX75R,JBL Tune 680 NC Kabellose On-Ear-Kopfhörer mit...,https://www.amazon.de/sspa/click?ie=UTF8&spc=M...,77.99,99.99,EUR,4.4,128,None,https://m.media-amazon.com/images/I/61wuPFetlI...,...,False,False,False,2,None,None,99.99,None,None,False
2,B0BTYCRJSS,soundcore by Anker P20i Kabellose Bluetooth Ko...,https://www.amazon.de/soundcore-Kabellose-Anpa...,19.99,27.99,EUR,4.3,108353,None,https://m.media-amazon.com/images/I/5181ILcyQJ...,...,True,False,False,3,None,None,27.99,None,None,False
3,B0DHL93XCN,"JBL Wave Beam 2, Kabellose Bluetooth-In-Ear-Ko...",https://www.amazon.de/JBL-Ear-Kopfh%C3%B6rer-N...,49.00,79.99,EUR,4.2,7427,None,https://m.media-amazon.com/images/I/61DFgTmj9x...,...,False,False,False,4,None,None,79.99,None,None,False
4,B0BTJD6LCL,Sony WH-CH520 Kabellose Bluetooth-Kopfhörer - ...,https://www.amazon.de/Sony-WH-CH520-Kabellose-...,29.00,39.99,EUR,4.5,43479,None,https://m.media-amazon.com/images/I/61rFE093es...,...,False,False,False,5,None,None,39.99,None,None,False


## Display all column names

In [13]:
print("Raw DataFrame columns:")

for column in products_raw_df.columns:
    print("-", column)

Raw DataFrame columns:
- asin
- title
- url
- price
- price_strikethrough
- currency
- rating
- reviews_count
- sales_volume
- image
- is_prime
- is_amazons_choice
- is_sponsored
- best_seller
- organic_position
- sponsored_position
- shipping_information
- highest_price
- pricing_count
- manufacturer
- is_video


## Check the DataFrame structure

In [14]:
products_raw_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 22 entries, 0 to 21
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   asin                  22 non-null     str    
 1   title                 22 non-null     str    
 2   url                   22 non-null     str    
 3   price                 22 non-null     float64
 4   price_strikethrough   20 non-null     float64
 5   currency              22 non-null     str    
 6   rating                22 non-null     float64
 7   reviews_count         22 non-null     int64  
 8   sales_volume          0 non-null      object 
 9   image                 22 non-null     str    
 10  is_prime              22 non-null     bool   
 11  is_amazons_choice     22 non-null     bool   
 12  is_sponsored          22 non-null     bool   
 13  best_seller           22 non-null     bool   
 14  organic_position      22 non-null     int64  
 15  sponsored_position    0 non-null    

## Check the DataFrame structure

In [15]:
products_raw_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 22 entries, 0 to 21
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   asin                  22 non-null     str    
 1   title                 22 non-null     str    
 2   url                   22 non-null     str    
 3   price                 22 non-null     float64
 4   price_strikethrough   20 non-null     float64
 5   currency              22 non-null     str    
 6   rating                22 non-null     float64
 7   reviews_count         22 non-null     int64  
 8   sales_volume          0 non-null      object 
 9   image                 22 non-null     str    
 10  is_prime              22 non-null     bool   
 11  is_amazons_choice     22 non-null     bool   
 12  is_sponsored          22 non-null     bool   
 13  best_seller           22 non-null     bool   
 14  organic_position      22 non-null     int64  
 15  sponsored_position    0 non-null    

## Standardise Column Names 
Because API column names can differ slightly, we will use a function that searches for possible column names.

## Create a helper function

In [20]:
def find_existing_column(dataframe, possible_names):
    """
    Return the first matching column from a list of possible names.
    """
    normalized_columns = {
        column.lower(): column
        for column in dataframe.columns
    }

    for name in possible_names:
        if name.lower() in normalized_columns:
            return normalized_columns[name.lower()]

    return None

## Detect useful business columns

In [21]:
detected_columns = {
    "asin": find_existing_column(
        products_raw_df,
        ["asin", "product_asin", "id"]
    ),
    "product_title": find_existing_column(
        products_raw_df,
        ["title", "name", "product_title"]
    ),
    "current_price": find_existing_column(
        products_raw_df,
        [
            "price",
            "current_price",
            "price.value",
            "price.amount",
            "prices.current_price"
        ]
    ),
    "original_price": find_existing_column(
        products_raw_df,
        [
            "original_price",
            "list_price",
            "rrp",
            "old_price",
            "price.before_price",
            "prices.original_price"
        ]
    ),
    "rating": find_existing_column(
        products_raw_df,
        [
            "rating",
            "stars",
            "review_rating",
            "rating.value"
        ]
    ),
    "review_count": find_existing_column(
        products_raw_df,
        [
            "reviews",
            "review_count",
            "reviews_count",
            "ratings_total",
            "total_reviews"
        ]
    ),
    "product_url": find_existing_column(
        products_raw_df,
        [
            "url",
            "product_url",
            "link"
        ]
    ),
    "image_url": find_existing_column(
        products_raw_df,
        [
            "image",
            "image_url",
            "thumbnail",
            "image.src"
        ]
    ),
    "search_position": find_existing_column(
        products_raw_df,
        [
            "position",
            "search_position",
            "rank"
        ]
    ),
    "is_sponsored": find_existing_column(
        products_raw_df,
        [
            "sponsored",
            "is_sponsored"
        ]
    ),
    "is_prime": find_existing_column(
        products_raw_df,
        [
            "prime",
            "is_prime",
            "amazon_prime"
        ]
    )
}

detected_columns_df = pd.DataFrame(
    detected_columns.items(),
    columns=["standard_name", "detected_api_column"]
)

detected_columns_df

,standard_name,detected_api_column
0,asin,asin
1,product_title,title
2,current_price,price
3,original_price,NaN
4,rating,rating
5,review_count,reviews_count
6,product_url,url
7,image_url,image
8,search_position,NaN
9,is_sponsored,is_sponsored


## Create a safe extraction function

In [23]:
def get_column_or_default(dataframe, column_name, default_value=np.nan):
    """
    Return a DataFrame column if it exists,
    otherwise return a default Series.
    """
    if pd.notna(column_name):
        return dataframe[column_name]

    return pd.Series(
        [default_value] * len(dataframe),
        index=dataframe.index
    )

## Create the working DataFrame

In [24]:
products_df = pd.DataFrame({
    "asin": get_column_or_default(
        products_raw_df,
        detected_columns["asin"]
    ),
    "product_title": get_column_or_default(
        products_raw_df,
        detected_columns["product_title"]
    ),
    "current_price_raw": get_column_or_default(
        products_raw_df,
        detected_columns["current_price"]
    ),
    "original_price_raw": get_column_or_default(
        products_raw_df,
        detected_columns["original_price"]
    ),
    "rating_raw": get_column_or_default(
        products_raw_df,
        detected_columns["rating"]
    ),
    "review_count_raw": get_column_or_default(
        products_raw_df,
        detected_columns["review_count"]
    ),
    "product_url": get_column_or_default(
        products_raw_df,
        detected_columns["product_url"]
    ),
    "image_url": get_column_or_default(
        products_raw_df,
        detected_columns["image_url"]
    ),
    "search_position": get_column_or_default(
        products_raw_df,
        detected_columns["search_position"]
    ),
    "is_sponsored": get_column_or_default(
        products_raw_df,
        detected_columns["is_sponsored"],
        False
    ),
    "is_prime": get_column_or_default(
        products_raw_df,
        detected_columns["is_prime"],
        False
    )
})

products_df.head()

,asin,product_title,current_price_raw,original_price_raw,rating_raw,review_count_raw,product_url,image_url,search_position,is_sponsored,is_prime
0,B0FSH42VZW,JBL Tune 730 BT Wireless Over-Ear-Kopfhörermit...,59.99,NaN,4.4,463,https://www.amazon.de/sspa/click?ie=UTF8&spc=M...,https://m.media-amazon.com/images/I/61WyJZWEhz...,NaN,False,False
1,B0FV3VX75R,JBL Tune 680 NC Kabellose On-Ear-Kopfhörer mit...,77.99,NaN,4.4,128,https://www.amazon.de/sspa/click?ie=UTF8&spc=M...,https://m.media-amazon.com/images/I/61wuPFetlI...,NaN,False,False
2,B0BTYCRJSS,soundcore by Anker P20i Kabellose Bluetooth Ko...,19.99,NaN,4.3,108353,https://www.amazon.de/soundcore-Kabellose-Anpa...,https://m.media-amazon.com/images/I/5181ILcyQJ...,NaN,False,False
3,B0DHL93XCN,"JBL Wave Beam 2, Kabellose Bluetooth-In-Ear-Ko...",49.00,NaN,4.2,7427,https://www.amazon.de/JBL-Ear-Kopfh%C3%B6rer-N...,https://m.media-amazon.com/images/I/61DFgTmj9x...,NaN,False,False
4,B0BTJD6LCL,Sony WH-CH520 Kabellose Bluetooth-Kopfhörer - ...,29.00,NaN,4.5,43479,https://www.amazon.de/Sony-WH-CH520-Kabellose-...,https://m.media-amazon.com/images/I/61rFE093es...,NaN,False,False


## Clean Price, Rating and Review Values

In [27]:
## Create a numeric cleaning function

In [26]:
def clean_numeric_value(value):
    """
    Convert values such as:
    '€49.99'
    '4.5 out of 5 stars'
    '1,234'
    into numeric values.
    """
    if pd.isna(value):
        return np.nan

    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value)

    value_text = str(value).strip()

    value_text = value_text.replace("\xa0", "")
    value_text = value_text.replace("€", "")
    value_text = value_text.replace("$", "")
    value_text = value_text.replace("£", "")
    value_text = value_text.replace(" ", "")

    number_match = re.search(
        r"-?\d+(?:[.,]\d+)?",
        value_text
    )

    if not number_match:
        return np.nan

    number_text = number_match.group()

    if "," in number_text and "." in number_text:
        if number_text.rfind(",") > number_text.rfind("."):
            number_text = (
                number_text
                .replace(".", "")
                .replace(",", ".")
            )
        else:
            number_text = number_text.replace(",", "")

    elif "," in number_text:
        decimal_part = number_text.split(",")[-1]

        if len(decimal_part) == 2:
            number_text = number_text.replace(",", ".")
        else:
            number_text = number_text.replace(",", "")

    return pd.to_numeric(
        number_text,
        errors="coerce"
    )

## Clean the numeric columns

In [29]:
products_df["current_price"] = (
    products_df["current_price_raw"]
    .apply(clean_numeric_value)
)

products_df["original_price"] = (
    products_df["original_price_raw"]
    .apply(clean_numeric_value)
)

products_df["rating"] = (
    products_df["rating_raw"]
    .apply(clean_numeric_value)
)

products_df["review_count"] = (
    products_df["review_count_raw"]
    .apply(clean_numeric_value)
)

products_df["search_position"] = pd.to_numeric(
    products_df["search_position"],
    errors="coerce"
)

products_df[
    [
        "current_price_raw",
        "current_price",
        "rating_raw",
        "rating",
        "review_count_raw",
        "review_count"
    ]
].head(10)

,current_price_raw,current_price,rating_raw,rating,review_count_raw,review_count
0,59.99,59.99,4.4,4.4,463,463.0
1,77.99,77.99,4.4,4.4,128,128.0
2,19.99,19.99,4.3,4.3,108353,108353.0
3,49.00,49.00,4.2,4.2,7427,7427.0
4,29.00,29.00,4.5,4.5,43479,43479.0
5,28.49,28.49,4.6,4.6,67076,67076.0
6,10.99,10.99,4.4,4.4,1242,1242.0
7,35.66,35.66,4.3,4.3,374,374.0
8,25.64,25.64,4.3,4.3,22976,22976.0
9,13.29,13.29,4.5,4.5,2209,2209.0


## Convert review count to a nullable integer

In [30]:
products_df["review_count"] = (
    products_df["review_count"]
    .round()
    .astype("Int64")
)

products_df["search_position"] = (
    products_df["search_position"]
    .round()
    .astype("Int64")
)

## Clean product titles and ASIN values (Text and Boolean Fields)

In [31]:
products_df["asin"] = (
    products_df["asin"]
    .astype("string")
    .str.strip()
)

products_df["product_title"] = (
    products_df["product_title"]
    .astype("string")
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

## Standardise Boolean values

In [32]:
def clean_boolean(value):
    if pd.isna(value):
        return False

    if isinstance(value, bool):
        return value

    value_text = str(value).strip().lower()

    return value_text in {
        "true",
        "1",
        "yes",
        "y",
        "prime",
        "sponsored"
    }


products_df["is_prime"] = (
    products_df["is_prime"]
    .apply(clean_boolean)
)

products_df["is_sponsored"] = (
    products_df["is_sponsored"]
    .apply(clean_boolean)
)

## Remove Invalid and Duplicate Records
 ## Check missing values before cleaning

In [33]:
missing_before = (
    products_df
    .isna()
    .sum()
    .sort_values(ascending=False)
)

missing_before

original_price_raw    22
original_price        22
search_position       22
current_price_raw      0
rating_raw             0
product_title          0
asin                   0
product_url            0
review_count_raw       0
is_sponsored           0
image_url              0
is_prime               0
current_price          0
rating                 0
review_count           0
dtype: int64

## Remove rows without a useful title

In [34]:
rows_before_cleaning = len(products_df)

products_df = products_df[
    products_df["product_title"].notna()
].copy()

products_df = products_df[
    products_df["product_title"].str.len() > 0
].copy()

print("Rows before cleaning:", rows_before_cleaning)
print("Rows after removing missing titles:", len(products_df))

Rows before cleaning: 22
Rows after removing missing titles: 22


## Remove duplicate products

In [35]:
products_df["duplicate_key"] = (
    products_df["asin"]
    .fillna(products_df["product_title"])
)

duplicate_count = products_df.duplicated(
    subset=["duplicate_key"]
).sum()

products_df = products_df.drop_duplicates(
    subset=["duplicate_key"],
    keep="first"
).copy()

print("Duplicate records found:", duplicate_count)
print("Rows after duplicate removal:", len(products_df))

Duplicate records found: 1
Rows after duplicate removal: 21


## Create Business Calculated Fields 
## Create the currency and marketplace columns

In [36]:
products_df["currency"] = "EUR"
products_df["marketplace"] = "Amazon Germany"
products_df["search_query"] = "wireless headphones"

## Create discount percentage

In [38]:
products_df["discount_percentage"] = np.where(
    (
        products_df["original_price"].notna()
        & products_df["current_price"].notna()
        & (products_df["original_price"] > 0)
        & (
            products_df["original_price"]
            > products_df["current_price"]
        )
    ),
    (
        (
            products_df["original_price"]
            - products_df["current_price"]
        )
        / products_df["original_price"]
        * 100
    ),
    0
)

products_df["discount_percentage"] = (
    products_df["discount_percentage"]
    .round(2)
)

## Create price categories

In [39]:
products_df["price_category"] = pd.cut(
    products_df["current_price"],
    bins=[
        -np.inf,
        30,
        60,
        120,
        np.inf
    ],
    labels=[
        "Budget",
        "Mid-range",
        "Premium",
        "Luxury"
    ]
)

## Create rating categories

In [41]:
products_df["rating_category"] = pd.cut(
    products_df["rating"],
    bins=[
        -np.inf,
        3.5,
        4.0,
        4.5,
        5.0
    ],
    labels=[
        "Low",
        "Average",
        "Good",
        "Excellent"
    ],
    include_lowest=True
)

## Add the extraction timestamp

In [42]:
products_df["extraction_timestamp"] = datetime.now()

products_df["extraction_date"] = (
    products_df["extraction_timestamp"]
    .dt.date
)

## Create a Brand Field
Amazon search results may not always provide a separate brand field. We will first detect whether the raw data includes one.

## Check for a brand column 

In [43]:
brand_column = find_existing_column(
    products_raw_df,
    [
        "brand",
        "manufacturer",
        "product_brand"
    ]
)

print("Detected brand column:", brand_column)

Detected brand column: manufacturer


## Create the brand field

In [46]:
known_brands = [
    "JBL",
    "Sony",
    "Soundcore",
    "Anker",
    "Bose",
    "Sennheiser",
    "Apple",
    "Beats",
    "Samsung",
    "Philips",
    "Logitech",
    "Jabra",
    "Skullcandy",
    "Marshall",
    "Xiaomi",
    "Huawei",
    "Fachixy",
    "ZZU"
]

brand_pattern = "|".join(
    re.escape(brand)
    for brand in known_brands
)

# Extract brand directly from the product title
products_df["brand"] = (
    products_df["product_title"]
    .str.extract(
        rf"(?i)\b({brand_pattern})\b",
        expand=False
    )
)

# Standardise selected brand names
brand_standardisation = {
    "jbl": "JBL",
    "sony": "Sony",
    "soundcore": "Soundcore",
    "anker": "Anker",
    "bose": "Bose",
    "sennheiser": "Sennheiser",
    "apple": "Apple",
    "beats": "Beats",
    "samsung": "Samsung",
    "philips": "Philips",
    "logitech": "Logitech",
    "jabra": "Jabra",
    "skullcandy": "Skullcandy",
    "marshall": "Marshall",
    "xiaomi": "Xiaomi",
    "huawei": "Huawei",
    "fachixy": "Fachixy",
    "zzu": "ZZU"
}

products_df["brand"] = (
    products_df["brand"]
    .str.lower()
    .map(brand_standardisation)
    .fillna("Unknown")
)

products_df[
    ["product_title", "brand"]
].head(15)

,product_title,brand
0,JBL Tune 730 BT Wireless Over-Ear-Kopfhörermit...,JBL
1,JBL Tune 680 NC Kabellose On-Ear-Kopfhörer mit...,JBL
2,soundcore by Anker P20i Kabellose Bluetooth Ko...,Soundcore
3,"JBL Wave Beam 2, Kabellose Bluetooth-In-Ear-Ko...",JBL
4,Sony WH-CH520 Kabellose Bluetooth-Kopfhörer - ...,Sony
5,soundcore by Anker Q20i kabelloser Bluetooth O...,Soundcore
6,"Bluetooth Kopfhörer, In Ear Kopfhörer Kabellos...",Samsung
7,"Fachixy FC100 Headset mit Mikrofon, 2,4G Wirel...",Fachixy
8,"Bluetooth 5.4 Kopfhörer Sport, 75Std 2026 Kopf...",Unknown
9,"ZZU Bluetooth Kopfhörer, Kopfhörer Kabellos Bl...",ZZU


## Select Final Columns

## Create the final cleaned dataset 

In [48]:
final_columns = [
    "asin",
    "product_title",
    "brand",
    "current_price",
    "original_price",
    "currency",
    "discount_percentage",
    "price_category",
    "rating",
    "rating_category",
    "review_count",
    "search_position",
    "is_prime",
    "is_sponsored",
    "marketplace",
    "search_query",
    "product_url",
    "image_url",
    "extraction_date",
    "extraction_timestamp"
]

cleaned_products_df = products_df[
    final_columns
].copy()

cleaned_products_df.head()

,asin,product_title,brand,current_price,original_price,currency,discount_percentage,price_category,rating,rating_category,review_count,search_position,is_prime,is_sponsored,marketplace,search_query,product_url,image_url,extraction_date,extraction_timestamp
0,B0FSH42VZW,JBL Tune 730 BT Wireless Over-Ear-Kopfhörermit...,JBL,59.99,NaN,EUR,0.0,Mid-range,4.4,Good,463,<NA>,False,False,Amazon Germany,wireless headphones,https://www.amazon.de/sspa/click?ie=UTF8&spc=M...,https://m.media-amazon.com/images/I/61WyJZWEhz...,2026-06-17,2026-06-17 15:54:20.345120
1,B0FV3VX75R,JBL Tune 680 NC Kabellose On-Ear-Kopfhörer mit...,JBL,77.99,NaN,EUR,0.0,Premium,4.4,Good,128,<NA>,False,False,Amazon Germany,wireless headphones,https://www.amazon.de/sspa/click?ie=UTF8&spc=M...,https://m.media-amazon.com/images/I/61wuPFetlI...,2026-06-17,2026-06-17 15:54:20.345120
2,B0BTYCRJSS,soundcore by Anker P20i Kabellose Bluetooth Ko...,Soundcore,19.99,NaN,EUR,0.0,Budget,4.3,Good,108353,<NA>,False,False,Amazon Germany,wireless headphones,https://www.amazon.de/soundcore-Kabellose-Anpa...,https://m.media-amazon.com/images/I/5181ILcyQJ...,2026-06-17,2026-06-17 15:54:20.345120
3,B0DHL93XCN,"JBL Wave Beam 2, Kabellose Bluetooth-In-Ear-Ko...",JBL,49.00,NaN,EUR,0.0,Mid-range,4.2,Good,7427,<NA>,False,False,Amazon Germany,wireless headphones,https://www.amazon.de/JBL-Ear-Kopfh%C3%B6rer-N...,https://m.media-amazon.com/images/I/61DFgTmj9x...,2026-06-17,2026-06-17 15:54:20.345120
4,B0BTJD6LCL,Sony WH-CH520 Kabellose Bluetooth-Kopfhörer - ...,Sony,29.00,NaN,EUR,0.0,Budget,4.5,Good,43479,<NA>,False,False,Amazon Germany,wireless headphones,https://www.amazon.de/Sony-WH-CH520-Kabellose-...,https://m.media-amazon.com/images/I/61rFE093es...,2026-06-17,2026-06-17 15:54:20.345120


## Check the final dataset structure

In [49]:
print("Final dataset shape:", cleaned_products_df.shape)

cleaned_products_df.info()

Final dataset shape: (21, 20)
<class 'pandas.DataFrame'>
Index: 21 entries, 0 to 21
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   asin                  21 non-null     string        
 1   product_title         21 non-null     string        
 2   brand                 21 non-null     str           
 3   current_price         21 non-null     float64       
 4   original_price        0 non-null      float64       
 5   currency              21 non-null     str           
 6   discount_percentage   21 non-null     float64       
 7   price_category        21 non-null     category      
 8   rating                21 non-null     float64       
 9   rating_category       21 non-null     category      
 10  review_count          21 non-null     Int64         
 11  search_position       0 non-null      Int64         
 12  is_prime              21 non-null     bool          
 13  is_spons

## Check final missing values 

In [51]:
final_missing_values = (
    cleaned_products_df
    .isna()
    .sum()
    .sort_values(ascending=False)
)

final_missing_values

search_position         21
original_price          21
asin                     0
product_title            0
current_price            0
brand                    0
discount_percentage      0
currency                 0
price_category           0
rating                   0
rating_category          0
review_count             0
is_prime                 0
is_sponsored             0
marketplace              0
search_query             0
product_url              0
image_url                0
extraction_date          0
extraction_timestamp     0
dtype: int64

## Check Final Missing Values

In [52]:
final_missing_values = (
    cleaned_products_df
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print("Missing values in final cleaned dataset:")
display(final_missing_values.to_frame(name="missing_count"))

Missing values in final cleaned dataset:


,missing_count
search_position,21
original_price,21
asin,0
product_title,0
current_price,0
brand,0
discount_percentage,0
currency,0
price_category,0
rating,0


## Check Missing Value Percentages

In [53]:
missing_summary_df = pd.DataFrame({
    "missing_count": cleaned_products_df.isna().sum(),
    "missing_percentage": (
        cleaned_products_df.isna().mean() * 100
    ).round(2)
}).sort_values(
    by="missing_count",
    ascending=False
)

display(missing_summary_df)

,missing_count,missing_percentage
search_position,21,100.0
original_price,21,100.0
asin,0,0.0
product_title,0,0.0
current_price,0,0.0
brand,0,0.0
discount_percentage,0,0.0
currency,0,0.0
price_category,0,0.0
rating,0,0.0


## Run Data Quality Checks

In [54]:
quality_checks = {
    "total_rows": len(cleaned_products_df),

    "total_columns": cleaned_products_df.shape[1],

    "duplicate_asins": (
        cleaned_products_df["asin"]
        .dropna()
        .duplicated()
        .sum()
    ),

    "missing_titles": (
        cleaned_products_df["product_title"]
        .isna()
        .sum()
    ),

    "missing_current_prices": (
        cleaned_products_df["current_price"]
        .isna()
        .sum()
    ),

    "negative_prices": (
        cleaned_products_df["current_price"] < 0
    ).sum(),

    "zero_prices": (
        cleaned_products_df["current_price"] == 0
    ).sum(),

    "missing_ratings": (
        cleaned_products_df["rating"]
        .isna()
        .sum()
    ),

    "ratings_above_5": (
        cleaned_products_df["rating"] > 5
    ).sum(),

    "ratings_below_0": (
        cleaned_products_df["rating"] < 0
    ).sum(),

    "negative_review_counts": (
        cleaned_products_df["review_count"] < 0
    ).sum(),

    "missing_asins": (
        cleaned_products_df["asin"]
        .isna()
        .sum()
    ),

    "missing_brands": (
        cleaned_products_df["brand"]
        .isna()
        .sum()
    )
}

quality_checks_df = pd.DataFrame(
    quality_checks.items(),
    columns=["quality_check", "result"]
)

display(quality_checks_df)

,quality_check,result
0,total_rows,21
1,total_columns,20
2,duplicate_asins,0
3,missing_titles,0
4,missing_current_prices,0
5,negative_prices,0
6,zero_prices,0
7,missing_ratings,0
8,ratings_above_5,0
9,ratings_below_0,0


## Add Data Quality Status

In [55]:
quality_checks_df["status"] = quality_checks_df.apply(
    lambda row: (
        "PASS"
        if row["quality_check"] in [
            "total_rows",
            "total_columns"
        ]
        or row["result"] == 0
        else "CHECK"
    ),
    axis=1
)

display(quality_checks_df)

,quality_check,result,status
0,total_rows,21,PASS
1,total_columns,20,PASS
2,duplicate_asins,0,PASS
3,missing_titles,0,PASS
4,missing_current_prices,0,PASS
5,negative_prices,0,PASS
6,zero_prices,0,PASS
7,missing_ratings,0,PASS
8,ratings_above_5,0,PASS
9,ratings_below_0,0,PASS


## Validate Required Columns

In [56]:
required_columns = [
    "asin",
    "product_title",
    "brand",
    "current_price",
    "currency",
    "price_category",
    "rating",
    "rating_category",
    "review_count",
    "is_prime",
    "is_sponsored",
    "marketplace",
    "search_query",
    "product_url",
    "image_url",
    "extraction_date",
    "extraction_timestamp"
]

missing_required_columns = [
    column
    for column in required_columns
    if column not in cleaned_products_df.columns
]

if missing_required_columns:
    raise ValueError(
        f"Missing required columns: {missing_required_columns}"
    )

print("All required columns are present.")

All required columns are present.


## Validate Required Data

In [57]:
required_non_null_columns = [
    "asin",
    "product_title",
    "brand",
    "current_price",
    "rating",
    "review_count"
]

required_null_counts = (
    cleaned_products_df[
        required_non_null_columns
    ]
    .isna()
    .sum()
)

display(
    required_null_counts.to_frame(
        name="missing_required_values"
    )
)

if required_null_counts.sum() > 0:
    raise ValueError(
        "Required fields still contain missing values."
    )

print("All required fields contain valid values.")

,missing_required_values
asin,0
product_title,0
brand,0
current_price,0
rating,0
review_count,0


All required fields contain valid values.


## Remove Temporary or Unnecessary Columns

In [58]:
columns_to_remove = [
    "duplicate_key",
    "current_price_raw",
    "original_price_raw",
    "rating_raw",
    "review_count_raw"
]

cleaned_products_df = cleaned_products_df.drop(
    columns=[
        column
        for column in columns_to_remove
        if column in cleaned_products_df.columns
    ],
    errors="ignore"
)

print("Temporary columns removed successfully.")
print("Final number of columns:", cleaned_products_df.shape[1])

Temporary columns removed successfully.
Final number of columns: 20


## Round Numeric Values

In [59]:
cleaned_products_df["current_price"] = (
    cleaned_products_df["current_price"]
    .round(2)
)

cleaned_products_df["original_price"] = (
    cleaned_products_df["original_price"]
    .round(2)
)

cleaned_products_df["discount_percentage"] = (
    cleaned_products_df["discount_percentage"]
    .round(2)
)

cleaned_products_df["rating"] = (
    cleaned_products_df["rating"]
    .round(2)
)

print("Numeric values rounded successfully.")

Numeric values rounded successfully.


## Reorder the Dataset Safely

In [61]:
final_column_order = [
    "asin",
    "product_title",
    "brand",
    "current_price",
    "original_price",
    "currency",
    "discount_percentage",
    "price_category",
    "rating",
    "rating_category",
    "review_count",
    "search_position",
    "is_prime",
    "is_sponsored",
    "marketplace",
    "search_query",
    "product_url",
    "image_url",
    "extraction_date",
    "extraction_timestamp"
]

cleaned_products_df = cleaned_products_df[
    final_column_order
].copy()

print("Columns reordered successfully.")
display(cleaned_products_df.head())

Columns reordered successfully.


,asin,product_title,brand,current_price,original_price,currency,discount_percentage,price_category,rating,rating_category,review_count,search_position,is_prime,is_sponsored,marketplace,search_query,product_url,image_url,extraction_date,extraction_timestamp
0,B0FSH42VZW,JBL Tune 730 BT Wireless Over-Ear-Kopfhörermit...,JBL,59.99,NaN,EUR,0.0,Mid-range,4.4,Good,463,<NA>,False,False,Amazon Germany,wireless headphones,https://www.amazon.de/sspa/click?ie=UTF8&spc=M...,https://m.media-amazon.com/images/I/61WyJZWEhz...,2026-06-17,2026-06-17 15:54:20.345120
1,B0FV3VX75R,JBL Tune 680 NC Kabellose On-Ear-Kopfhörer mit...,JBL,77.99,NaN,EUR,0.0,Premium,4.4,Good,128,<NA>,False,False,Amazon Germany,wireless headphones,https://www.amazon.de/sspa/click?ie=UTF8&spc=M...,https://m.media-amazon.com/images/I/61wuPFetlI...,2026-06-17,2026-06-17 15:54:20.345120
2,B0BTYCRJSS,soundcore by Anker P20i Kabellose Bluetooth Ko...,Soundcore,19.99,NaN,EUR,0.0,Budget,4.3,Good,108353,<NA>,False,False,Amazon Germany,wireless headphones,https://www.amazon.de/soundcore-Kabellose-Anpa...,https://m.media-amazon.com/images/I/5181ILcyQJ...,2026-06-17,2026-06-17 15:54:20.345120
3,B0DHL93XCN,"JBL Wave Beam 2, Kabellose Bluetooth-In-Ear-Ko...",JBL,49.00,NaN,EUR,0.0,Mid-range,4.2,Good,7427,<NA>,False,False,Amazon Germany,wireless headphones,https://www.amazon.de/JBL-Ear-Kopfh%C3%B6rer-N...,https://m.media-amazon.com/images/I/61DFgTmj9x...,2026-06-17,2026-06-17 15:54:20.345120
4,B0BTJD6LCL,Sony WH-CH520 Kabellose Bluetooth-Kopfhörer - ...,Sony,29.00,NaN,EUR,0.0,Budget,4.5,Good,43479,<NA>,False,False,Amazon Germany,wireless headphones,https://www.amazon.de/Sony-WH-CH520-Kabellose-...,https://m.media-amazon.com/images/I/61rFE093es...,2026-06-17,2026-06-17 15:54:20.345120


## Check Final Dataset Summary

In [62]:
print("Final dataset shape:", cleaned_products_df.shape)
print("\nFinal column names:")

for column in cleaned_products_df.columns:
    print("-", column)

print("\nFinal data types:")
print(cleaned_products_df.dtypes)

Final dataset shape: (21, 20)

Final column names:
- asin
- product_title
- brand
- current_price
- original_price
- currency
- discount_percentage
- price_category
- rating
- rating_category
- review_count
- search_position
- is_prime
- is_sponsored
- marketplace
- search_query
- product_url
- image_url
- extraction_date
- extraction_timestamp

Final data types:
asin                            string
product_title                   string
brand                              str
current_price                  float64
original_price                 float64
currency                           str
discount_percentage            float64
price_category                category
rating                         float64
rating_category               category
review_count                     Int64
search_position                  Int64
is_prime                          bool
is_sponsored                      bool
marketplace                        str
search_query                       str
product_ur

## Display Final Business Columns

In [64]:
display_columns = [
    "asin",
    "product_title",
    "brand",
    "current_price",
    "price_category",
    "rating",
    "rating_category",
    "review_count",
    "is_prime",
    "is_sponsored"
]

display(
    cleaned_products_df[
        display_columns
    ].head(15)
)

,asin,product_title,brand,current_price,price_category,rating,rating_category,review_count,is_prime,is_sponsored
0,B0FSH42VZW,JBL Tune 730 BT Wireless Over-Ear-Kopfhörermit...,JBL,59.99,Mid-range,4.4,Good,463,False,False
1,B0FV3VX75R,JBL Tune 680 NC Kabellose On-Ear-Kopfhörer mit...,JBL,77.99,Premium,4.4,Good,128,False,False
2,B0BTYCRJSS,soundcore by Anker P20i Kabellose Bluetooth Ko...,Soundcore,19.99,Budget,4.3,Good,108353,False,False
3,B0DHL93XCN,"JBL Wave Beam 2, Kabellose Bluetooth-In-Ear-Ko...",JBL,49.00,Mid-range,4.2,Good,7427,False,False
4,B0BTJD6LCL,Sony WH-CH520 Kabellose Bluetooth-Kopfhörer - ...,Sony,29.00,Budget,4.5,Good,43479,False,False
5,B0C3HCD34R,soundcore by Anker Q20i kabelloser Bluetooth O...,Soundcore,28.49,Budget,4.6,Excellent,67076,False,False
6,B0FY2JJRB2,"Bluetooth Kopfhörer, In Ear Kopfhörer Kabellos...",Samsung,10.99,Budget,4.4,Good,1242,False,False
7,B0FDL13R3J,"Fachixy FC100 Headset mit Mikrofon, 2,4G Wirel...",Fachixy,35.66,Mid-range,4.3,Good,374,False,False
8,B0GLG2J9RM,"Bluetooth 5.4 Kopfhörer Sport, 75Std 2026 Kopf...",Unknown,25.64,Budget,4.3,Good,22976,False,False
9,B0GZGGDJ67,"ZZU Bluetooth Kopfhörer, Kopfhörer Kabellos Bl...",ZZU,13.29,Budget,4.5,Good,2209,False,False


## Save the Cleaned CSV File

In [65]:
cleaned_csv_file = (
    PROCESSED_DATA_DIR
    / "amazon_wireless_headphones_cleaned.csv"
)

cleaned_products_df.to_csv(
    cleaned_csv_file,
    index=False,
    encoding="utf-8-sig"
)

print("Cleaned CSV saved successfully.")
print("Saved file:", cleaned_csv_file)

Cleaned CSV saved successfully.
Saved file: C:\Users\srika\Documents\Srikanth_Amazon_BI_Project\data\processed\amazon_wireless_headphones_cleaned.csv


## Save the Excel File 

In [66]:
cleaned_excel_file = (
    PROCESSED_DATA_DIR
    / "amazon_wireless_headphones_cleaned.xlsx"
)

cleaned_products_df.to_excel(
    cleaned_excel_file,
    index=False,
    engine="openpyxl"
)

print("Cleaned Excel file saved successfully.")
print("Saved file:", cleaned_excel_file)

Cleaned Excel file saved successfully.
Saved file: C:\Users\srika\Documents\Srikanth_Amazon_BI_Project\data\processed\amazon_wireless_headphones_cleaned.xlsx


## Verify the Saved Files

In [67]:
file_verification = pd.DataFrame({
    "file_name": [
        cleaned_csv_file.name,
        cleaned_excel_file.name
    ],
    "exists": [
        cleaned_csv_file.exists(),
        cleaned_excel_file.exists()
    ],
    "size_bytes": [
        cleaned_csv_file.stat().st_size
        if cleaned_csv_file.exists()
        else 0,

        cleaned_excel_file.stat().st_size
        if cleaned_excel_file.exists()
        else 0
    ]
})

display(file_verification)

,file_name,exists,size_bytes
0,amazon_wireless_headphones_cleaned.csv,True,20028
1,amazon_wireless_headphones_cleaned.xlsx,True,10424


## Reopen the Saved CSV

In [68]:
verification_df = pd.read_csv(
    cleaned_csv_file,
    encoding="utf-8-sig"
)

print("Saved CSV loaded successfully.")
print("Rows:", verification_df.shape[0])
print("Columns:", verification_df.shape[1])

display(verification_df.head())

Saved CSV loaded successfully.
Rows: 21
Columns: 20


,asin,product_title,brand,current_price,original_price,currency,discount_percentage,price_category,rating,rating_category,review_count,search_position,is_prime,is_sponsored,marketplace,search_query,product_url,image_url,extraction_date,extraction_timestamp
0,B0FSH42VZW,JBL Tune 730 BT Wireless Over-Ear-Kopfhörermit...,JBL,59.99,NaN,EUR,0.0,Mid-range,4.4,Good,463,NaN,False,False,Amazon Germany,wireless headphones,https://www.amazon.de/sspa/click?ie=UTF8&spc=M...,https://m.media-amazon.com/images/I/61WyJZWEhz...,2026-06-17,2026-06-17 15:54:20.345120
1,B0FV3VX75R,JBL Tune 680 NC Kabellose On-Ear-Kopfhörer mit...,JBL,77.99,NaN,EUR,0.0,Premium,4.4,Good,128,NaN,False,False,Amazon Germany,wireless headphones,https://www.amazon.de/sspa/click?ie=UTF8&spc=M...,https://m.media-amazon.com/images/I/61wuPFetlI...,2026-06-17,2026-06-17 15:54:20.345120
2,B0BTYCRJSS,soundcore by Anker P20i Kabellose Bluetooth Ko...,Soundcore,19.99,NaN,EUR,0.0,Budget,4.3,Good,108353,NaN,False,False,Amazon Germany,wireless headphones,https://www.amazon.de/soundcore-Kabellose-Anpa...,https://m.media-amazon.com/images/I/5181ILcyQJ...,2026-06-17,2026-06-17 15:54:20.345120
3,B0DHL93XCN,"JBL Wave Beam 2, Kabellose Bluetooth-In-Ear-Ko...",JBL,49.00,NaN,EUR,0.0,Mid-range,4.2,Good,7427,NaN,False,False,Amazon Germany,wireless headphones,https://www.amazon.de/JBL-Ear-Kopfh%C3%B6rer-N...,https://m.media-amazon.com/images/I/61DFgTmj9x...,2026-06-17,2026-06-17 15:54:20.345120
4,B0BTJD6LCL,Sony WH-CH520 Kabellose Bluetooth-Kopfhörer - ...,Sony,29.00,NaN,EUR,0.0,Budget,4.5,Good,43479,NaN,False,False,Amazon Germany,wireless headphones,https://www.amazon.de/Sony-WH-CH520-Kabellose-...,https://m.media-amazon.com/images/I/61rFE093es...,2026-06-17,2026-06-17 15:54:20.345120


## Compare Saved CSV with the DataFrame

In [69]:
print(
    "Row count matches:",
    len(verification_df) == len(cleaned_products_df)
)

print(
    "Column count matches:",
    verification_df.shape[1]
    == cleaned_products_df.shape[1]
)

print(
    "All column names match:",
    list(verification_df.columns)
    == list(cleaned_products_df.columns)
)

Row count matches: True
Column count matches: True
All column names match: True


## Create a Cleaning Summary

In [70]:
cleaning_summary = pd.DataFrame({
    "metric": [
        "Raw product records",
        "Duplicate records removed",
        "Final unique products",
        "Final columns",
        "Missing original prices",
        "Missing search positions",
        "Products with valid prices",
        "Products with valid ratings"
    ],
    "value": [
        22,
        1,
        len(cleaned_products_df),
        cleaned_products_df.shape[1],
        cleaned_products_df["original_price"].isna().sum(),
        cleaned_products_df["search_position"].isna().sum(),
        cleaned_products_df["current_price"].notna().sum(),
        cleaned_products_df["rating"].notna().sum()
    ]
})

display(cleaning_summary)

,metric,value
0,Raw product records,22
1,Duplicate records removed,1
2,Final unique products,21
3,Final columns,20
4,Missing original prices,21
5,Missing search positions,21
6,Products with valid prices,21
7,Products with valid ratings,21


In [72]:
import os

os.startfile(PROCESSED_DATA_DIR)